In [15]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class HelloState(TypedDict):
    name: str
    greeting: str

# define a node
def greet(state: HelloState) -> HelloState:
    name = state['name']
    return {"greeting": f"Hello, {name}!"}

def add_emoji(state: HelloState) -> HelloState:
    greeting = state['greeting']
    return {"greeting": f"{greeting} 🚀"}

# define a graph
graph = StateGraph(HelloState)

graph.add_node("greet", greet)
graph.add_node("add_emoji", add_emoji)

graph.add_edge(START, "greet")
graph.add_edge("greet", "add_emoji")
graph.add_edge("add_emoji", END)

# compile the graph
app = graph.compile()

# run the graph
result = app.invoke({"name": "World"})
print(result["greeting"])
from IPython.display import Image, display
try:
    display(Image(app.get_graph(xray=True).draw_png()))
except Exception as e:
    print("Error displaying graph: ", e)

Hello, World! 🚀
Error displaying graph:  Install pygraphviz to draw graphs: `pip install pygraphviz`.


In [22]:
# case2. weather recommendation

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END


class WeatherState(TypedDict):
    temperature: float
    recommendation: str

def check_temperature(state: WeatherState) -> dict:
    return {"temperature": 20.0}

def route_by_temperature(state: WeatherState) -> Literal["cold", "warm", "hot"]:
    temperature = state["temperature"]
    if temperature < 15:
        return "cold"
    elif temperature < 25:
        return "warm"
    else:
        return "hot"

def recommend_cold(state: WeatherState) -> dict:
    return {"recommendation": "穿厚外套"}

def recommend_warm(state: WeatherState) -> dict:
    return {"recommendation": "穿短袖"}

def recommend_hot(state: WeatherState) -> dict:
    return {"recommendation": "多喝水，穿清凉衣服"}

# create graph（独立 StateGraph，避免与上面 Hello 示例共用变量 graph；重复运行本 cell 也不会重复注册节点）
graph = StateGraph(WeatherState)
graph.add_node("check", check_temperature)
graph.add_node("cold", recommend_cold)
graph.add_node("warm", recommend_warm)
graph.add_node("hot", recommend_hot)

graph.add_edge(START, "check")

graph.add_conditional_edges(
    "check",
    route_by_temperature,
    {"cold": "cold", "warm": "warm", "hot": "hot"}
)

for node in ["cold", "warm", "hot"]:
    graph.add_edge(node, END)

# compile the graph
app = graph.compile()

# Test the graph
result = app.invoke({})
print(result["recommendation"])

穿短袖


In [6]:
import os
from sre_parse import State
from typing import TypedDict, Literal
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage


# define State
class AgentModerationState(TypedDict):
    content: str
    analysis: str
    decision: str # approve | rejected | needs_review
    reason: str
    confidence: float

# define model
model = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.0,
    base_url = os.getenv("base_url"),
    api_key = os.getenv("api_key"),
    max_tokens=None,
    max_retries=2,
)

# define nodes
def analyze_content(state: AgentModerationState) -> dict:
    content = state["content"]
    system_prompt = """
    你是一个内容审核专家，请对以下内容进行审核，并给出审核结果。
    审核标准：
    1. 内容是否违反相关法律法规
    2. 内容是否含有敏感信息
    3. 内容是否含有不当言论

    返回严格的json格式：
    {
        "has_issues": true/false,
        "issues": ["issue1", "issue2"],
        "severity": "low/medium/high",
        "confidence": 0.0-1.0
    }
    """
    # 使用模型分析内容
    response = model.invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=content),
        ]
    )
    return {"analysis": response.content}

def make_agent_decision(state: AgentModerationState) -> dict:
    analysis = state["analysis"]
    content = state["content"]
    system_prompt = """
        你是一个最终审核专家，你将根据我提供的原文内容和分析结果，给出最终的审核结果。
        以严格的json格式返回：
        {
            "decision": "approve" | "rejected" | "needs_review",
            "reason": "简短说明",
            "confidence": 0.0-1.0
        }
    
    """
    response = model.invoke(
        [
            SystemMessage(system_prompt),
            HumanMessage(f"原内容为：{content}, 分析为：{analysis}")
        ]
    )

    import json
    try:
        result = json.loads(response.content)
        return {
            "decision": result["decision"],
            "reason": result["reason"],
            "confidence": result["confidence"],
        }
    except:
        # 返回默认的决策结果，表示无法解析模型回应
        return {
            "decision": "needs_review",
            "reason": "无法解析模型回应",
            "confidence": 0.0,
        }

# router func
def should_auto_decide(state: AgentModerationState) -> Literal["decide", "review"]:
    if "high" in state.get("analysis", "").lower():
        return "review"
    return "decide"

def human_review(state: AgentModerationState) -> dict:
    # 这里是人工审核的占位函数，可根据需要实现人工审核逻辑
    return {
        "decision": "needs_review",
        "reason": "等待人工审核",
        "confidence": 1.0,
    }

# create graph
Agent_graph = StateGraph(AgentModerationState)

Agent_graph.add_node("analyze_content", analyze_content)
Agent_graph.add_node("make_agent_decision", make_agent_decision)
Agent_graph.add_node("should_auto_decide", should_auto_decide)
Agent_graph.add_node("human_review", human_review)

Agent_graph.add_edge(START, "analyze_content")
Agent_graph.add_conditional_edges(
    "analyze_content",
    should_auto_decide,
    {"review": "human_review", "decide": "make_agent_decision"}

)
for node in ["human_review", "make_agent_decision"]:
    Agent_graph.add_edge(node, END)


#compile
app = Agent_graph.compile()

result = app.invoke({"content": "南昌天气是什么"})
print(result)

{'content': '南昌天气是什么', 'analysis': '{\n    "has_issues": false,\n    "issues": [],\n    "severity": "low",\n    "confidence": 0.95\n}', 'decision': 'approve', 'reason': '内容无问题', 'confidence': 0.95}
